# Transfermarkt Scrapper

In [28]:
# requirements:
# pip install requests beautifulsoup4 pandas lxml tqdm

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------ CONFIG ------------------
BASE = "https://www.transfermarkt.com"

LEAGUES = [
    {"name": "Premier League", "path": "premier-league", "code": "GB1", "extra": "leihe=1&intern=0&intern=1"},
    {"name": "Bundesliga", "path": "bundesliga", "code": "L1", "extra": "leihe=1&intern=0&intern=1"},
    {"name": "La Liga", "path": "laliga", "code": "ES1", "extra": "leihe=1&intern=0&intern=1"},
    {"name": "Serie A", "path": "serie-a", "code": "IT1", "extra": "leihe=3&intern=0&intern=1"},
    {"name": "Ligue 1", "path": "ligue-1", "code": "FR1", "extra": "leihe=1&intern=0&intern=1"},
]

SEASONS = [2020, 2021, 2022, 2023, 2024, 2026]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,es;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)


def clean_text(x):
    return re.sub(r"\s+", " ", x).strip() if x else ""


def get_soup(url):
    r = session.get(url, timeout=25)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")


# ------------------ SCRAPING ------------------
all_rows = []

for season in tqdm(SEASONS, desc="Seasons"):
    for lg in tqdm(LEAGUES, desc=f"Scraping Season {season}", leave=False):
        league_name, path, code, extra = lg["name"], lg["path"], lg["code"], lg["extra"]
        url = f"{BASE}/{path}/transfers/wettbewerb/{code}/plus/?saison_id={season}&s_w=&{extra}"

        try:
            soup = get_soup(url)
        except Exception as e:
            tqdm.write(f"⚠️ Skipping {league_name} {season}: {e}")
            continue

        for block in soup.select("div.responsive-table"):
            table = block.find("table")
            if not table:
                continue

            thead = table.find("thead")
            if not thead:
                continue
            first_th = clean_text(thead.find("th").get_text()) if thead.find("th") else ""
            if first_th.lower() != "in":  # only incoming transfers
                continue

            # Get club name
            to_team = ""
            prev_h2 = block.find_previous_sibling(lambda t: t.name == "h2")
            if prev_h2:
                to_team = clean_text(prev_h2.get_text())
            if not to_team:
                a_inside = block.select_one('a[title]')
                if a_inside and a_inside.get("title"):
                    to_team = clean_text(a_inside["title"])
            if not to_team:
                to_team = "UNKNOWN"

            tbody = table.find("tbody")
            if not tbody:
                continue

            for tr in tbody.find_all("tr"):
                tds = tr.find_all("td")
                if len(tds) < 8:
                    continue

                # Extract main fields
                player_a = tds[0].find("a")
                player = clean_text(player_a.get_text()) if player_a else clean_text(tds[0].get_text())
                if not player:
                    continue

                age = clean_text(tds[1].get_text())

                nat_img = tds[2].find("img")
                nation = nat_img["title"] if nat_img and nat_img.has_attr("title") else ""

                position = clean_text(tds[3].get_text())

                value = clean_text(tds[5].get_text())

                from_cell = tds[7]
                from_flag = from_cell.find("img")
                from_league = from_flag["title"] if from_flag and from_flag.has_attr("title") else ""
                from_team_a = from_cell.find("a")
                from_team = clean_text(from_team_a.get_text()) if from_team_a else ""

                all_rows.append(
                    {
                        "season": season,
                        "league": league_name,
                        "player": player,
                        "from_team": from_team,
                        "to_team": to_team,
                        "from_league": from_league,
                        "to_league": league_name,
                        "nation": nation,
                        "age": age,
                        "position": position,
                        "value": value,
                    }
                )

# ------------------ BUILD DATAFRAME ------------------
df = pd.DataFrame(all_rows)
print(f"\n✅ Total transfers scraped: {len(df)}")
print(df.head(20))
# df.to_csv("eu_top5_incoming_transfers.csv", index=False)


Seasons: 100%|██████████| 6/6 [02:13<00:00, 22.25s/it]


✅ Total transfers scraped: 9129
    season          league                  player      from_team  \
0     2020  Premier League           Thomas Partey       Atlético   
1     2020  Premier League       Gabriel Magalhães          Lille   
2     2020  Premier League              Pablo Marí       Flamengo   
3     2020  Premier League    Rúnar Alex Rúnarsson          Dijon   
4     2020  Premier League         Martin Ødegaard    Real Madrid   
5     2020  Premier League           Cédric Soares    Southampton   
6     2020  Premier League                 Willian        Chelsea   
7     2020  Premier League             Mathew Ryan       Brighton   
8     2020  Premier League        Emile Smith Rowe    Arsenal U23   
9     2020  Premier League             Dejan Iliev     Shrewsbury   
10    2020  Premier League             Dejan Iliev    Jagiellonia   
11    2020  Premier League      Henrikh Mkhitaryan           Roma   
12    2020  Premier League  Ainsley Maitland-Niles      West Brom   
1

In [29]:
df.to_csv("eu_top5_incoming_transfers.csv", index=False)